In [15]:
import sys
sys.path.append("../")
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
import numpy as np
from tqdm import tqdm
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict
from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.config.config import data_settings
import os
import h5py
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [16]:
def delete_all_files(directory):
    # Check if the directory exists
    if not os.path.exists(directory):
        print(f"The directory {directory} does not exist.")
        return
    
    # Iterate over all files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        try:
            # Check if it's a file (not a subdirectory)
            if os.path.isfile(file_path):
                os.remove(file_path)  # Delete the file
                print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")


In [17]:
def db_to_hdf5_files_single_file():
    """
    This version creates ONE h5 file for training, ONE for testing, ONE for validation,
    each containing chunked, resizable datasets ('features' and 'labels'),
    with a tqdm progress bar for each data split.
    """
    # You can also pull this from data_settings if you wish
    batch_file_size = 5096  
    num_bitboards = len(sample_bitboard_dict.keys())

    sets = {
        data_settings.TrainingDirectory: GamePositionRollup.is_training_data.is_(True),
        data_settings.TestingDirectory: GamePositionRollup.is_testing_data.is_(True),
        data_settings.ValidationDirectory: GamePositionRollup.is_validation_data.is_(True),
    }

    for h5_dir, filter_conditions in sets.items():

        delete_all_files(h5_dir)
        single_file_path = os.path.join(h5_dir, "data_all.h5")

        # Count total records in this split
        with next(get_db()) as session:
            total_records = session.query(GamePositionRollup)\
                                   .filter(filter_conditions)\
                                   .count()
            print(f"[{h5_dir}] Total records: {total_records}")

        if total_records == 0:
            print(f"No records found for {h5_dir}, skipping.")
            continue

        with h5py.File(single_file_path, 'w') as h5f:
            # Create resizable, chunked datasets
            features_dset = h5f.create_dataset(
                "features",
                shape=(0, num_bitboards, 8, 8),
                maxshape=(None, num_bitboards, 8, 8),
                dtype="uint64",
                chunks=(1, num_bitboards, 8, 8),
                compression="gzip"
            )
            labels_dset = h5f.create_dataset(
                "labels",
                shape=(0, 3),
                maxshape=(None, 3),
                dtype="uint64",
                chunks=(1, 3),
                compression="gzip"
            )

            current_size = 0

            with next(get_db()) as session:
                # Wrap the main loop with tqdm
                with tqdm(
                    total=total_records, 
                    desc=f"[{h5_dir}] Writing Records", 
                    unit=" records"
                ) as pbar:
                    for batch_start in range(0, total_records, batch_file_size):

                        records = (session.query(GamePositionRollup)
                                  .filter(filter_conditions)
                                  .offset(batch_start)
                                  .limit(batch_file_size)
                                  .all())

                        if not records:
                            break

                        # Collect batch data
                        features_list = []
                        labels_list = []

                        for record in records:
                            # Extract features
                            bitboard_values = [getattr(record, attr) 
                                               for attr in sample_bitboard_dict.keys()]
                            features = bitboards_to_array(bitboard_values)

                            # Extract labels
                            labels = record.win_buckets

                            features_list.append(features)
                            labels_list.append(labels)

                        # Convert to NumPy arrays
                        features_array = np.array(features_list, dtype=np.float32)
                        labels_array   = np.array(labels_list, dtype=np.float32)

                        # Resize HDF5 datasets
                        batch_size = features_array.shape[0]
                        new_size = current_size + batch_size

                        features_dset.resize((new_size, num_bitboards, 8, 8))
                        labels_dset.resize((new_size, 3))

                        # Write the new data
                        features_dset[current_size:new_size, ...] = features_array
                        labels_dset[current_size:new_size, ...]   = labels_array

                        current_size = new_size

                        # Update the tqdm progress bar
                        pbar.update(batch_size)

        print(f"Finished writing dataset to {single_file_path}")

In [18]:
db_to_hdf5_files_single_file()

Deleted: ./src/model/data/training\data_all.h5
[./src/model/data/training] Total records: 2230333


[./src/model/data/training] Writing Records: 100%|███████████████████████████████████████████████████████████████████████████████| 2230333/2230333 [06:59<00:00, 5314.70 records/s]


Finished writing dataset to ./src/model/data/training\data_all.h5
Deleted: ./src/model/data/testing\data_0.h5
[./src/model/data/testing] Total records: 46431


[./src/model/data/testing] Writing Records: 100%|████████████████████████████████████████████████████████████████████████████████████| 46431/46431 [00:09<00:00, 4675.27 records/s]


Finished writing dataset to ./src/model/data/testing\data_all.h5
Deleted: ./src/model/data/validation\data_0.h5
[./src/model/data/validation] Total records: 46464


[./src/model/data/validation] Writing Records: 100%|█████████████████████████████████████████████████████████████████████████████████| 46464/46464 [00:10<00:00, 4464.10 records/s]

Finished writing dataset to ./src/model/data/validation\data_all.h5


In [5]:
class HDF5SingleFileDataset(Dataset):
    """
    Reads from a single .h5 file containing:
      - dataset 'features' of shape (N, 12, 8, 8)
      - dataset 'labels' of shape (N, 3)
    in chunked, optionally compressed form.
    """

    def __init__(self, h5_path, transform=None):
        """
        Args:
            h5_path (str): Path to the .h5 file (e.g. 'training/data_all.h5').
            transform (callable, optional): Apply transformations to the features.
        """
        super().__init__()
        self.h5_path = h5_path
        self.transform = transform

        # Open file once to read total length
        with h5py.File(self.h5_path, 'r') as h5f:
            self.data_len = len(h5f['features'])  # or h5f['features'].shape[0]

    def __len__(self):
        return self.data_len

    def __getitem__(self, idx):
        # Ensure we open the file in read-only mode in each __getitem__
        # to avoid concurrency issues with DataLoader workers.
        with h5py.File(self.h5_path, 'r') as h5f:
            features = h5f['features'][idx]  # shape (12, 8, 8)
            labels = h5f['labels'][idx]      # shape (3,)

        # Optionally apply transforms
        if self.transform:
            features = self.transform(features)

        # Convert to torch Tensors
        features_tensor = torch.from_numpy(features)  # float32
        labels_tensor   = torch.from_numpy(labels)    # float32 (or int, depends on your usage)

        return features_tensor, labels_tensor


In [6]:
def get_dataloader(h5_path, batch_size=32, shuffle=True, num_workers=0):
    dataset = HDF5SingleFileDataset(h5_path=h5_path, transform=None)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)
    return loader




In [8]:

train_loader = get_dataloader(data_settings.TrainingDirectory, batch_size=64, shuffle=True, num_workers=4)
for epoch in range(num_epochs):
    for features, labels in train_loader:
        print(f"feature shape: {features.shape}, labels shape: {labels.shape}")
        # features => shape (64, 12, 8, 8)
        # labels   => shape (64, 3)
        # your training logic here...

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'training/data_all.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)